In [1]:
import numpy as np 
import pandas as pd 
import os
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. List files in the default input directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
# 2. Download the official Titanic dataset files using kagglehub
data_dir = kagglehub.competition_download("titanic")
print(f"Data successfully synced to path: {data_dir}")

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv
/kaggle/input/datasets/yasserh/titanic-dataset/Titanic-Dataset.csv
Data successfully synced to path: /kaggle/input/competitions/titanic


In [2]:
# Load the train and test CSV files from the synced directory path
train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape: {test_df.shape}")
train_df.head()

Train Dataset Shape: (891, 12)
Test Dataset Shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
import os

# 1. Safely reload raw datasets from the saved directory to restore dropped columns
train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))

# 2. Define the structural feature engineering logic
def engineer_features(df):
    # Extract operational social titles from Name strings
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    
    # Calculate unified family size metrics
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # Drop unique string columns to prevent overfitting
    return df.drop(['Name', 'Ticket', 'Cabin'], axis=1)

# 3. Apply steps to fresh copies of data
train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

# 4. Separate features from target variables
X = train_df.drop(['PassengerId', 'Survived'], axis=1)
y = train_df['Survived']
X_test_final = test_df.drop(['PassengerId'], axis=1)

# 5. Construct operational transformers for numeric and category splits
num_features = ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']
cat_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'IsAlone']

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

print("Features successfully engineered without path conflicts!")

Features successfully engineered without path conflicts!


In [4]:
# 1. Split the training dataset for local validation (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Package the preprocessing steps together with the Random Forest Classifier
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

# 3. Train the model pipeline on the training subset
model_pipeline.fit(X_train, y_train)

# 4. Generate predictions on the local validation set
y_pred = model_pipeline.predict(X_val)

# 5. Output evaluation metrics to check performance
print(f"Validation Accuracy Score: {accuracy_score(y_val, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_val, y_pred))

Validation Accuracy Score: 0.8324

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.87       110
           1       0.81      0.74      0.77        69

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179



In [5]:
# 1. Predict survival labels using the final holdout test dataframe
final_predictions = model_pipeline.predict(X_test_final)

# 2. Create the exact submission dataframe layout required by Kaggle
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": final_predictions
})

# 3. Save the dataframe to a CSV file in the current working directory
submission.to_csv('submission.csv', index=False)

print("File 'submission.csv' generated successfully!")

File 'submission.csv' generated successfully!
